# Big Query to download Ethereum data
-  The following code has been taken from the kaggle notebook as reference.
- A service account needs to be created.
- bigquery and db-mbs library needs to be installed.

In [4]:
# %pip install db-dtypes
# %pip install google-cloud-bigquery

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\User\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.19.5
    Uninstalling protobuf-3.19.5:
      Successfully uninstalled protobuf-3.19.5
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
web3 5.31.3 requires protobuf==3.19.5, but you have protobuf 4.22.0 which is incompatible.
You should consider upgrading via the 'c:\Users\User\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [2]:
from google.cloud import bigquery
import pandas as pd
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "C:/Users/User/Desktop/blockchain_research/eth-dataset-372206-0985b468a014.json"
client = bigquery.Client()

# Query by Allen Day, GooglCloud Developer Advocate (https://medium.com/@allenday)
query = """
SELECT 
  SUM(value/POWER(10,18)) AS sum_tx_ether,
  AVG(gas_price*(receipt_gas_used/POWER(10,18))) AS avg_tx_gas_cost,
  DATE(timestamp) AS tx_date
FROM
  `bigquery-public-data.crypto_ethereum.transactions` AS transactions,
  `bigquery-public-data.crypto_ethereum.blocks` AS blocks
WHERE TRUE
  AND transactions.block_number = blocks.number
  AND receipt_status = 1
  AND value > 0
GROUP BY tx_date
HAVING tx_date >= '2015-01-01' AND tx_date <= '2023-12-31'
ORDER BY tx_date
"""
query_job = client.query(query)

iterator = query_job.result(timeout=90)
rows = list(query_job)

# Transform the rows into a nice pandas dataframe
df = pd.DataFrame(data=[list(x.values()) for x in rows], columns=list(rows[0].keys()))

# Look at the first 10
df.tail(3)

,sum_tx_ether,avg_tx_gas_cost,tx_date
1952,7.666137e+05,0.002499,2023-02-19
1953,1.130323e+06,0.003400,2023-02-20
1954,9.458600e+05,0.002744,2023-02-21


In [3]:
df.head(3)

,sum_tx_ether,avg_tx_gas_cost,tx_date
0,6.176871e+06,0.000592,2017-10-16
1,1.109204e+07,0.000464,2017-10-17
2,1.459824e+07,0.000494,2017-10-18


# Block table and the attributes from 2015 to 2018
- The following query explores all the attributes of the block

In [19]:
query1 = """ 
SELECT 
  timestamp,
  number AS block_number,
  'hash' as hsh,
  parent_hash,
  nonce,
  miner,
  difficulty,
  size,
  gas_limit,
  gas_used,
  transaction_count
  base_fee_per_gas
  
FROM
  `bigquery-public-data.crypto_ethereum.blocks`
WHERE
  EXTRACT(YEAR FROM timestamp) = 2015
  # AND EXTRACT(YEAR FROM timestamp) <= 2018
ORDER BY timestamp
"""

In [22]:
# Transactions query


In [25]:
# Contracts query


In [27]:
# Token_transfer query


In [29]:
# token query


In [30]:
# Execute the query and convert the results to a Pandas DataFrame
# There is an issue with the 'hash' attribute in the above query, it produces a BadRequest error
# df2 = client.query(query1).to_dataframe()
# df2.head(3)
df2 = client.query(query5)
print(df2)

QueryJob<project=eth-dataset-372206, location=US, id=6b65a0ab-5a92-4da1-89e4-4c717f2daf4b>


- The following query works if we omit 'hash' attribute. 
- We need to fix the hash attribute error issue. 

In [20]:
# The following query does not have the 'hash' attributes to be queried. It Works!!!!
# query2 = """ 
# SELECT 
#   timestamp,
#   number AS block_number, 
#   parent_hash
#   nonce,
#   miner,
#   difficulty,
#   size,
#   gas_limit,
#   gas_used,
#   transaction_count
#   base_fee_per_gas
  
# FROM
#   `bigquery-public-data.crypto_ethereum.blocks`
# WHERE
#   EXTRACT(YEAR FROM timestamp) >= 2015
#   AND EXTRACT(YEAR FROM timestamp) <= 2018
# ORDER BY timestamp
# """


In [29]:

# Execute the query and convert the results to a Pandas DataFrame
df2 = client.query(query2).to_dataframe()
df2.head()

In [30]:
# Exporting the dataframe to a text file:
df2.to_csv("/Users/dr.rubaiyatislam/Documents/big_query/block.txt",sep='\t')

In [33]:
export_df = pd.read_csv('/Users/dr.rubaiyatislam/Documents/big_query/block.txt',sep='\t')
export_df.head(3)

,Unnamed: 0,timestamp,block_number,nonce,miner,difficulty,size,gas_limit,gas_used,base_fee_per_gas
0,0,2015-07-30 15:26:28+00:00,1,0x539bd4979fef1ec4,0x05a56e2d52c817161883f50c441c3228cfe54d9f,1.717148e+10,537,5000,0,0
1,1,2015-07-30 15:26:57+00:00,2,0xb853fa261a86aa9e,0xdd2f1e6e498202e86d8f5442af596580a4f03c2c,1.716310e+10,544,5000,0,0
2,2,2015-07-30 15:27:28+00:00,3,0x2e9344e0cbde83ce,0x5088d623ba0fcf0131e0897a91734a4d83596aa0,1.715472e+10,1079,5000,0,0


In [34]:
export_df.tail(3)

,Unnamed: 0,timestamp,block_number,nonce,miner,difficulty,size,gas_limit,gas_used,base_fee_per_gas
6988611,6988611,2018-12-31 23:59:22+00:00,6988612,0x58709a501293367d,0xea674fdde714fd979de3edf0f56aa9716b898ec8,2.512630e+15,15526,8000029,6855179,125
6988612,6988612,2018-12-31 23:59:38+00:00,6988613,0xc226e1c3296e7b0a,0x52bc44d5378309ee2abf1539bf71de1b7d7be3b5,2.512768e+15,5481,8000029,1654731,26
6988613,6988613,2018-12-31 23:59:42+00:00,6988614,0xddfb0738002e1ee4,0x829bd824b016326a401d083b33d092293333a830,2.514132e+15,21421,8007840,7717545,98


In [36]:
print(len(export_df))

6988614


In [37]:
export_df.columns


Index(['Unnamed: 0', 'timestamp', 'block_number', 'nonce', 'miner',
       'difficulty', 'size', 'gas_limit', 'gas_used', 'base_fee_per_gas'],
      dtype='object')